# 04 — Tree-Based Models: XGBoost, LightGBM & SHAP

**Goals:**
1. Train XGBoost and LightGBM with walk-forward CV
2. Compare OOS IC against baselines from notebook 03
3. Use SHAP to understand *which features* are driving predictions
4. Diagnose overfitting: if in-sample IC >> OOS IC, complexity is not paying off

**Expected finding:** Tree models may show modestly higher mean IC than Ridge,
but ICIR may be similar or lower. This is a completely valid and instructive result
— it shows the non-linear interactions are either weak or unstable across regimes.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as cfg
from src.models import XGBoostRanker, LightGBMRanker, WalkForwardRunner
from src.targets import align_features_targets
from src.metrics import rolling_ic, rolling_topk_spread, ic_summary, hit_rate
from src.plotting import (
    plot_rolling_ic, plot_topk_spread, plot_feature_importance, plot_ic_comparison
)
from src.validation import WalkForwardSplitter

PROCESSED = cfg.processed_dir()
print('OK')

In [ ]:
features = pd.read_parquet(PROCESSED / 'features.parquet')
targets  = pd.read_parquet(PROCESSED / 'targets.parquet')
X, y_all = align_features_targets(features, targets)
y_rank = y_all['rank_target']
print(f'Dataset: {X.shape}')

## 1. XGBoost Walk-Forward

In [ ]:
splitter = WalkForwardSplitter(
    min_train_months=cfg.get('validation.min_train_months', 24),
    test_months=cfg.get('validation.test_months', 3),
    gap_days=cfg.get('validation.gap_days', 5),
)

runner = WalkForwardRunner(X, y_rank, splitter=splitter)
xgb_params = cfg.load_config().get('models', {}).get('xgboost', {})
xgb_results = runner.run_model(XGBoostRanker, {'params': xgb_params}, target='rank')
print('\nXGBoost summary:', xgb_results.summary())

In [ ]:
xgb_ic = rolling_ic(xgb_results.all_oos_scores, xgb_results.all_oos_returns)
fig = plot_rolling_ic(xgb_ic, model_name='XGBoost')
plt.show()

## 2. LightGBM Walk-Forward

In [ ]:
lgb_params = cfg.load_config().get('models', {}).get('lightgbm', {})
lgb_results = runner.run_model(LightGBMRanker, {'params': lgb_params}, target='rank')
print('LightGBM summary:', lgb_results.summary())

## 3. Overfitting Diagnostic: In-Sample vs OOS IC

In [ ]:
# Compute in-sample IC for last fold (the largest training set)
last_fold = xgb_results.fold_results[-1]
print(f'Last fold:')
print(f'  OOS IC  : {last_fold.ic:.4f}')

# Train model on last fold's training data and evaluate IN-SAMPLE
from src.validation import WalkForwardSplitter
splits = list(splitter.split(X))
last_split = splits[-1]

X_train = X.iloc[last_split.train_idx]
y_train = y_rank.iloc[last_split.train_idx].dropna()
X_train = X_train.loc[y_train.index]

model = XGBoostRanker(params=xgb_params)
model.fit(X_train, y_train)
is_scores = pd.Series(model.predict(X_train), index=X_train.index)

from src.metrics import rolling_ic
is_ic = rolling_ic(is_scores, y_rank.loc[X_train.index]).mean()
print(f'  In-sample IC: {is_ic:.4f}')
print(f'  IS/OOS ratio: {is_ic / last_fold.ic:.1f}x')
print('')
print('If IS/OOS >> 2x, significant overfitting is present.')
print('For high-noise financial data, some ratio is expected (1.5–3x is common).')

## 4. SHAP Value Analysis

In [ ]:
try:
    import shap

    # Compute SHAP on a sample of test data from the last fold
    X_test_last = X.iloc[last_split.test_idx]
    shap_vals, X_sample = model.shap_values(X_test_last, sample_n=500)

    # SHAP summary plot
    print('SHAP summary (bee-swarm):')
    shap.summary_plot(shap_vals, X_sample, max_display=15, show=True)

    # Average |SHAP| per feature
    mean_abs_shap = pd.Series(
        np.abs(shap_vals).mean(axis=0),
        index=X_sample.columns,
    ).sort_values(ascending=False)

    print('\nTop 10 features by mean |SHAP|:')
    print(mean_abs_shap.head(10).to_string())

    fig = plot_feature_importance(mean_abs_shap, top_n=20, model_name='XGBoost (SHAP)')
    plt.show()

except ImportError:
    print('Install shap: pip install shap')
except Exception as e:
    print(f'SHAP error: {e}')

## 5. Average Feature Importance Across All Folds

In [ ]:
fold_importances = [
    f.feature_importance for f in xgb_results.fold_results
    if f.feature_importance is not None
]
if fold_importances:
    avg_imp = pd.concat(fold_importances, axis=1).mean(axis=1).sort_values(ascending=False)
    std_imp = pd.concat(fold_importances, axis=1).std(axis=1)

    print('Feature importance stability (std/mean — lower is more stable across folds):')
    stability = (std_imp / avg_imp).reindex(avg_imp.head(15).index)
    print(stability.round(3).to_string())

    fig = plot_feature_importance(avg_imp, top_n=20, model_name='XGBoost (avg across folds)')
    plt.show()

## 6. IC Comparison: Baselines vs Trees

In [ ]:
# Load baseline results if you saved them, or quick-rerun
from src.models import BaselineRankModel
ridge_results = runner.run_model(BaselineRankModel, target='rank')

ic_dict = {
    'Ridge':    rolling_ic(ridge_results.all_oos_scores, ridge_results.all_oos_returns),
    'XGBoost':  rolling_ic(xgb_results.all_oos_scores,  xgb_results.all_oos_returns),
    'LightGBM': rolling_ic(lgb_results.all_oos_scores,  lgb_results.all_oos_returns),
}

fig = plot_ic_comparison(ic_dict)
plt.show()

print('\nComparison:')
for name, ic_s in ic_dict.items():
    s = ic_summary(ic_s)
    print(f'  {name:<10}: IC={s["mean_ic"]:+.4f}  ICIR={s["icir"]:+.3f}  hit={s["pct_positive"]:.1%}')

In [ ]:
# Save best OOS scores for backtest notebook
# Select model with highest ICIR
all_results = {'ridge': ridge_results, 'xgboost': xgb_results, 'lightgbm': lgb_results}
best_name = max(all_results, key=lambda k: ic_summary(rolling_ic(
    all_results[k].all_oos_scores, all_results[k].all_oos_returns
)).get('icir', -999))

best_scores = all_results[best_name].all_oos_scores
best_scores.to_parquet(cfg.processed_dir() / 'oos_scores.parquet')
print(f'Best model: {best_name} — OOS scores saved to data/processed/oos_scores.parquet')

## Summary
Fill in after running:
- XGBoost IC: | ICIR:
- LightGBM IC: | ICIR:
- IS/OOS IC ratio (overfitting check):
- Top features by SHAP:
- Takeaway vs baseline:

→ Proceed to `05_evaluation.ipynb` for full walk-forward report